## ETL Pipeline for Retail Data

### 1. Extract Data

In [54]:
import pandas as pd
import sqlite3

# โหลดชุดข้อมูล
try:
    # เปลี่ยน encoding เป็น utf-8-sig เพื่อตัดอักขระ ï»¿ ออกจาก Sale_ID
    df = pd.read_csv('retail_logs.csv', encoding='utf-8-sig')

    #Data Cleansing ทำความสะอาด String (ตัดช่องว่างและปรับตัวพิมพ์) ---
    text_cols = ['Branch', 'Province', 'Region', 'Product_Name', 'Category']
    for col in text_cols:
        if col in df.columns:
            # ตัดช่องว่างหัวท้าย และทำให้เป็น Title Case (เช่น PHUKET -> Phuket)
            df[col] = df[col].astype(str).str.strip().str.title()

            # แปลง Nan, 'Nan', 'None' ให้กลับเป็น np.nan
            df[col] = df[col].replace('Nan', pd.NA)

    print("Data loaded and cleaned successfully.")
    display(df.head())
    print("\nDataFrame Info:")
    df.info()
except Exception as e:
    print(f"An error occurred: {e}")

Data loaded and cleaned successfully.


,Sale_ID,Store_Code,Branch,Province,Region,Product_Name,Category,Sale_Date,Quantity,Unit_Price,Discount_Percent
0,SALE-00264,KKN-01,Khon Kaen Center,Khon Kaen,Northeast,Cookie Box,Bakery,2026-03-15,4,150.0,10.0
1,SALE-00077,PKT-01,Phuket Town,Phuket,<NA>,Travel Mug,Merchandise,14-May-2026,4,320.0,5.0
2,SALE-00221,CBI-01,Bangsaen,Chonburi,East,Tote Bag,Merchandise,06-May-2026,6,180.0,15.0
3,SALE-00150,PKT-01,Phuket Town,Phuket,South,Caesar Salad,Food,26-Mar-2026,2,110.0,0.0
4,SALE-00084,PKT-01,Phuket Town,Phuket,South,Mineral Water,Beverage,01-May-2026,6,25.0,10.0



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Sale_ID           325 non-null    object 
 1   Store_Code        325 non-null    object 
 2   Branch            325 non-null    object 
 3   Province          325 non-null    object 
 4   Region            323 non-null    object 
 5   Product_Name      325 non-null    object 
 6   Category          325 non-null    object 
 7   Sale_Date         325 non-null    object 
 8   Quantity          325 non-null    int64  
 9   Unit_Price        325 non-null    float64
 10  Discount_Percent  324 non-null    float64
dtypes: float64(2), int64(1), object(8)
memory usage: 28.1+ KB


### 2. Transform Data to Star Schema


#### 2.1. Date Dimension

In [55]:
# แปลง 'Sale_Date' เป็นอ็อบเจกต์ datetime
df['Sale_Date'] = pd.to_datetime(df['Sale_Date'], errors='coerce')

# ลบแถวที่ Sale_Date เป็น NaT (หากมีการแปลงล้มเหลว)
df.dropna(subset=['Sale_Date'], inplace=True)

# สร้างตาราง Date Dimension
date_dimension = df[['Sale_Date']].drop_duplicates().copy()
date_dimension.rename(columns={'Sale_Date': 'full_date'}, inplace=True)
date_dimension['date_key'] = date_dimension['full_date'].dt.strftime('%Y%m%d').astype(int)
date_dimension['year'] = date_dimension['full_date'].dt.year
date_dimension['month'] = date_dimension['full_date'].dt.month
date_dimension['day'] = date_dimension['full_date'].dt.day
date_dimension['day_of_week'] = date_dimension['full_date'].dt.dayofweek # Monday=0, Sunday=6
date_dimension['day_name'] = date_dimension['full_date'].dt.day_name()
date_dimension['month_name'] = date_dimension['full_date'].dt.month_name()

date_dimension = date_dimension.sort_values('full_date').reset_index(drop=True)
display(date_dimension.head())

,full_date,date_key,year,month,day,day_of_week,day_name,month_name
0,2026-03-01,20260301,2026,3,1,6,Sunday,March
1,2026-03-03,20260303,2026,3,3,1,Tuesday,March
2,2026-03-04,20260304,2026,3,4,2,Wednesday,March
3,2026-03-05,20260305,2026,3,5,3,Thursday,March
4,2026-03-06,20260306,2026,3,6,4,Friday,March


#### 2.2. Product Dimension

In [56]:
# สร้างตาราง Product Dimension
product_dimension = df[['Product_Name', 'Category', 'Unit_Price']].copy()
product_dimension.dropna(subset=['Product_Name'], inplace=True)

#บังคับให้ Product_Name ไม่ซ้ำกัน (หากราคาต่างกัน ให้ยึดราคาสุดท้าย) ---
product_dimension = product_dimension.drop_duplicates(subset=['Product_Name'], keep='last')

product_dimension['product_key'] = product_dimension.reset_index(drop=True).index + 1
product_dimension = product_dimension[['product_key', 'Product_Name', 'Category', 'Unit_Price']]
display(product_dimension.head())

,product_key,Product_Name,Category,Unit_Price
216,1,Travel Mug,Merchandise,320.0
218,2,Tote Bag,Merchandise,180.0
226,3,Thai Milk Tea,Beverage,73.5
237,4,Cookie Box,Bakery,150.0
250,5,Caesar Salad,Food,110.0


#### 2.3. Location Dimension

In [57]:
# สร้างตาราง Location Dimension
location_dimension = df[['Branch', 'Province', 'Region']].copy()
location_dimension.dropna(subset=['Branch', 'Province'], inplace=True)

#บังคับให้ Branch และ Province ไม่ซ้ำกัน ---
location_dimension = location_dimension.drop_duplicates(subset=['Branch', 'Province'], keep='last')

location_dimension['location_key'] = location_dimension.reset_index(drop=True).index + 1
location_dimension = location_dimension[['location_key', 'Branch', 'Province', 'Region']]
display(location_dimension.head())

,location_key,Branch,Province,Region
256,1,Central Rama 9,Bangkok,Central
259,2,Nimman,Chiang Mai,North
268,3,Khon Kaen Center,Khon Kaen,Northeast
288,4,Pattaya,Chonburi,East
302,5,Rayong City,Rayong,East


#### 2.4. Sales Fact Table

In [58]:
sales_fact = df.copy()

sales_fact['date_key'] = sales_fact['Sale_Date'].dt.strftime('%Y%m%d').astype(int)

# ผสานข้อมูล
sales_fact = pd.merge(sales_fact, product_dimension[['Product_Name', 'product_key']], on='Product_Name', how='left')
sales_fact = pd.merge(sales_fact, location_dimension[['Branch', 'Province', 'location_key']], on=['Branch', 'Province'], how='left')

sales_fact['TotalAmount'] = sales_fact['Quantity'] * sales_fact['Unit_Price'] * (1 - sales_fact['Discount_Percent']/100)

# ชื่อ Sale_ID จะไม่มี ï»¿ แล้ว เพราะแก้ encoding
sales_fact = sales_fact[['Sale_ID', 'date_key', 'product_key', 'location_key', 'Quantity', 'Unit_Price', 'Discount_Percent', 'TotalAmount']]
sales_fact.rename(columns={'Sale_ID': 'InvoiceNo'}, inplace=True)

sales_fact.dropna(subset=['date_key', 'product_key', 'location_key'], inplace=True)
sales_fact['date_key'] = sales_fact['date_key'].astype(int)
sales_fact['product_key'] = sales_fact['product_key'].astype(int)
sales_fact['location_key'] = sales_fact['location_key'].astype(int)

display(sales_fact.head())
print(f"Sales Fact Table shape: {sales_fact.shape}")

,InvoiceNo,date_key,product_key,location_key,Quantity,Unit_Price,Discount_Percent,TotalAmount
0,SALE-00264,20260315,4,3,4,150.0,10.0,540.00
1,SALE-00271,20260628,3,8,1,70.0,0.0,70.00
2,SALE-00243,20260514,5,5,1,110.0,5.0,104.50
3,SALE-00215,20260415,6,3,1,25.0,15.0,21.25
4,SALE-00277,20260321,7,3,3,120.0,0.0,360.00


Sales Fact Table shape: (100, 8)


### 3. Load Data to SQLite Database

In [59]:
DATABASE_NAME = 'retail_warehouse.db'

# เชื่อมต่อกับฐานข้อมูล SQLite
conn = sqlite3.connect(DATABASE_NAME)
cursor = conn.cursor()

# โหลดตารางมิติ
date_dimension.to_sql('date', conn, if_exists='replace', index=False)
product_dimension.to_sql('product', conn, if_exists='replace', index=False)
location_dimension.to_sql('location', conn, if_exists='replace', index=False)

# โหลดตารางข้อเท็จจริง (sales)
sales_fact.to_sql('sales', conn, if_exists='replace', index=False)

# คอมมิตและปิดการเชื่อมต่อ
conn.commit()
conn.close()

print(f"Data successfully loaded into {DATABASE_NAME}")

# ตัวเลือกเสริม: ตรวจสอบตารางและข้อมูลบางส่วนจากตาราง sales
conn = sqlite3.connect(DATABASE_NAME)
print("\nTables in the database:")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

print("\nFirst 5 rows from sales table:")
sales_from_db = pd.read_sql_query("SELECT * FROM sales LIMIT 5;", conn)
display(sales_from_db)

conn.close()

Data successfully loaded into retail_warehouse.db

Tables in the database:
[('date',), ('product',), ('location',), ('sales',)]

First 5 rows from sales table:


,InvoiceNo,date_key,product_key,location_key,Quantity,Unit_Price,Discount_Percent,TotalAmount
0,SALE-00264,20260315,4,3,4,150.0,10.0,540.00
1,SALE-00271,20260628,3,8,1,70.0,0.0,70.00
2,SALE-00243,20260514,5,5,1,110.0,5.0,104.50
3,SALE-00215,20260415,6,3,1,25.0,15.0,21.25
4,SALE-00277,20260321,7,3,3,120.0,0.0,360.00
